# Prepare a ZincSight batch for Tamarind

Use this notebook to enter IDs or combine uploaded structures into one ready-to-upload ZIP. **ZincSight predictions run on Tamarind**, after you upload that ZIP; this notebook only downloads/prepares input files. Use a standard CPU Colab runtime. No API key, account connection, or GPU is needed.

1. Run **Enter inputs**, then paste IDs (commas, spaces, or one per line) and choose any upload options.
2. Run **Prepare ZIP**. If selected, upload ID lists (`.txt`, `.csv`, `.tsv`) and/or structures/archives when prompted.
3. Run **Download**, then choose the ZIP in **Batch archive or prepared ID batch** on the ZincSight Tamarind tool page.

PDB/assembly, UniProt, complete AlphaFold, MGYP, and complete TED domain IDs are supported. Examples: `1KLS`, `2A0S-assembly1`, `P12345`, `AF-P12345-F2-model_v4`, `AF-A0A287EUI1-F1-model_v4_TED01`. Files may be PDB/mmCIF/ENT (also gzip compressed), ZIP, TAR, TAR.GZ, or TGZ. Limits: **100 structures; 5 MB each; 100 MB total uncompressed structure data**. The ZIP includes the source IDs/URLs and structure hashes.

The preparer source is embedded in this notebook; it does not fetch or execute remote code. For detailed input/output behavior, see the [integration README](https://github.com/MECHTI1/ZincSight/blob/feature/tamarind-integration/integrations/tamarind/README.md). Cite the ZincSight method: [Mechtinger et al., Protein Science (2025)](https://doi.org/10.1002/pro.70350).


In [ ]:
#@title 1. Enter inputs { display-mode: "form" }
import hashlib
from pathlib import Path
import types
import ipywidgets as widgets
from IPython.display import display

# Exact standalone preparer source, bundled so no remote code is executed.
PREPARER_SOURCE = '#!/usr/bin/env python3\n"""Prepare Colab-style inputs before uploading them to offline Tamarind jobs.\n\nThis companion CLI runs on the user\'s networked computer, not inside Tamarind.\nIt accepts public structure IDs, ID-list files, structures, folders, and bounded\narchives, then produces a ZIP and a provenance manifest. It never reads an API\nkey, submits a job, or includes credentials or absolute local paths in outputs.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport re\nimport stat\nimport tarfile\nimport tempfile\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nimport zipfile\n\nMAX_STRUCTURES = 100\nMAX_FILE_BYTES = 5_000_000\nMAX_TOTAL_BYTES = 100_000_000\nMAX_ARCHIVE_BYTES = 200_000_000\nMAX_ARCHIVE_ENTRIES = 1000\nMAX_ID_FILE_BYTES = 1_000_000\nPROVIDER_PREFIXES = {\n    "files.rcsb.org": "/download/",\n    "alphafold.ebi.ac.uk": "/files/",\n    "api.esmatlas.com": "/fetchPredictedStructure/",\n    "ted.cathdb.info": "/api/v1/files/",\n}\nUNIPROT = r"(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})"\nAF_MODEL = rf"AF-({UNIPROT})-F([1-9][0-9]*)-model_v([1-9][0-9]*)"\nID_HEADERS = {"id", "ids", "identifier", "identifiers", "structure_id", "structure_ids",\n              "query_id", "pdb", "pdb_id", "uniprot", "uniprot_id", "accession"}\n\n\ndef parse_ids(text: str) -> list[str]:\n    """Accept comma/whitespace-separated identifiers and optional # comments."""\n    clean = "\\n".join(line.split("#", 1)[0] for line in text.splitlines())\n    return list(dict.fromkeys(part for part in re.split(r"[\\s,]+", clean.strip()) if part))\n\n\ndef read_id_list(data: bytes, suffix: str = ".txt") -> list[str]:\n    if not data or len(data) > MAX_ID_FILE_BYTES:\n        raise ValueError("ID-list files must be nonempty and at most 1 MB.")\n    text = data.decode("utf-8-sig")\n    if "\\x00" in text:\n        raise ValueError("An ID list must be a UTF-8 text file.")\n    if suffix.lower() == ".txt":\n        return parse_ids(text)\n    rows = list(csv.reader(io.StringIO(text), delimiter="\\t" if suffix.lower() == ".tsv" else ","))\n    rows = [row for row in rows if row and any(value.strip() for value in row)\n            and not row[0].lstrip().startswith("#")]\n    if not rows:\n        return []\n    header = [value.strip().lower().replace(" ", "_") for value in rows[0]]\n    index = next((i for i, value in enumerate(header) if value in ID_HEADERS), None)\n    if index is not None:\n        values = [row[index] for row in rows[1:] if len(row) > index]\n    else:\n        values = [value for row in rows for value in row]\n    return parse_ids("\\n".join(values))\n\n\ndef resolve_identifier(identifier: str) -> dict:\n    """Resolve identifiers only through fixed public-provider URL templates."""\n    value = identifier.strip()\n    pdb = re.fullmatch(r"([0-9][A-Z0-9]{3})(?:-assembly([1-9][0-9]{0,2}))?", value, re.I)\n    if pdb:\n        canonical = pdb[1].upper() + (f"-assembly{pdb[2]}" if pdb[2] else "")\n        return {"canonical_id": canonical, "source_url": f"https://files.rcsb.org/download/{canonical}.cif",\n                "file_format": "cif", "filename": canonical + ".cif"}\n    if re.fullmatch(r"MGYP[0-9]{12}", value, re.I):\n        canonical = value.upper()\n        return {"canonical_id": canonical,\n                "source_url": f"https://api.esmatlas.com/fetchPredictedStructure/{canonical}.pdb",\n                "file_format": "pdb", "filename": canonical + ".pdb"}\n    ted = re.fullmatch(AF_MODEL + r"_TED([0-9]{2,4})", value, re.I)\n    if ted and int(ted[4]) > 0:\n        canonical = f"AF-{ted[1].upper()}-F{ted[2]}-model_v{ted[3]}_TED{ted[4]}"\n        return {"canonical_id": canonical, "source_url": f"https://ted.cathdb.info/api/v1/files/{canonical}.pdb",\n                "file_format": "pdb", "filename": canonical + ".pdb"}\n    model = re.fullmatch(AF_MODEL, value, re.I)\n    if model:\n        canonical = f"AF-{model[1].upper()}-F{model[2]}-model_v{model[3]}"\n    elif re.fullmatch(UNIPROT, value, re.I):\n        canonical = f"AF-{value.upper()}-F1-model_v6"\n    else:\n        raise ValueError("Unsupported identifier; use a PDB/assembly, UniProt, full AlphaFold, MGYP or TED domain ID.")\n    return {"canonical_id": canonical, "source_url": f"https://alphafold.ebi.ac.uk/files/{canonical}.cif",\n            "file_format": "cif", "filename": canonical + ".cif"}\n\n\ndef validate_provider_url(url: str) -> None:\n    parsed = urllib.parse.urlsplit(url)\n    prefix = PROVIDER_PREFIXES.get(parsed.hostname or "")\n    if (parsed.scheme != "https" or prefix is None or not parsed.path.startswith(prefix)\n            or parsed.username is not None or parsed.password is not None\n            or parsed.port not in (None, 443) or parsed.query or parsed.fragment):\n        raise ValueError("Only fixed HTTPS structure-provider URLs are permitted.")\n\n\nclass ProviderRedirectHandler(urllib.request.HTTPRedirectHandler):\n    def redirect_request(self, request, fp, code, msg, headers, newurl):\n        validate_provider_url(newurl)\n        return super().redirect_request(request, fp, code, msg, headers, newurl)\n\n\ndef read_bounded(stream, limit: int, label: str) -> bytes:\n    data = stream.read(limit + 1)\n    if len(data) > limit:\n        raise ValueError(f"{label} exceeds its allowed byte limit.")\n    return data\n\n\ndef download_structure(url: str, *, max_bytes: int = MAX_FILE_BYTES, attempts: int = 3,\n                       timeout: int = 30, opener=None, sleep=time.sleep) -> bytes:\n    validate_provider_url(url)\n    if not 1 <= attempts <= 3 or not 1 <= timeout <= 60:\n        raise ValueError("Downloads allow 1–3 attempts and a 1–60 second timeout.")\n    opener = opener or urllib.request.build_opener(ProviderRedirectHandler())\n    for attempt in range(attempts):\n        try:\n            request = urllib.request.Request(url, headers={"User-Agent": "ZincSight-Tamarind-Input-Preparer/1.0"})\n            with opener.open(request, timeout=timeout) as response:\n                validate_provider_url(response.geturl())\n                length = response.headers.get("Content-Length")\n                if length is not None and int(length) > max_bytes:\n                    raise ValueError("Downloaded structure exceeds the allowed byte limit.")\n                if "text/html" in response.headers.get("Content-Type", "").lower():\n                    raise ValueError("The provider returned an HTML page instead of a structure.")\n                return read_bounded(response, max_bytes, "Downloaded structure")\n        except urllib.error.HTTPError as exc:\n            if exc.code not in (429, 500, 502, 503, 504) or attempt + 1 == attempts:\n                raise ValueError(f"Public structure provider returned HTTP {exc.code}.") from exc\n        except (urllib.error.URLError, TimeoutError) as exc:\n            if attempt + 1 == attempts:\n                raise ValueError("Public structure download failed after bounded retries.") from exc\n        sleep(0.25 * (2 ** attempt))\n    raise AssertionError("Unreachable download retry state")\n\n\ndef structure_format(name: str) -> str | None:\n    lower = name.lower()\n    if lower.endswith(".gz"):\n        lower = lower[:-3]\n    suffix = PurePosixPath(lower).suffix\n    return "pdb" if suffix in (".pdb", ".ent") else "cif" if suffix in (".cif", ".mmcif") else None\n\n\ndef structure_bytes(data: bytes, name: str, limit: int) -> bytes:\n    if name.lower().endswith(".gz"):\n        with gzip.GzipFile(fileobj=io.BytesIO(data)) as compressed:\n            data = read_bounded(compressed, limit, "Decompressed structure")\n    if not data or len(data) > limit:\n        raise ValueError("Each structure must be nonempty and at most 5 MB.")\n    text = data.decode("utf-8-sig")\n    if "\\x00" in text:\n        raise ValueError("Structures must be text PDB or mmCIF files.")\n    fmt = structure_format(name)\n    if fmt == "pdb" and not any(line.startswith("ATOM  ") for line in text.splitlines()):\n        raise ValueError(f"{PurePosixPath(name).name}: PDB input must contain protein ATOM records.")\n    if fmt == "cif" and "_atom_site." not in text:\n        raise ValueError(f"{PurePosixPath(name).name}: mmCIF input must contain an atom_site table.")\n    return data\n\n\ndef safe_archive_name(name: str) -> str:\n    clean = name.rstrip("/")\n    # Standard tar tools commonly emit ./member paths; normalize that prefix\n    # before detecting traversal and collisions, without allowing internal dots.\n    while clean.startswith("./"):\n        clean = clean[2:]\n    if (not clean or "\\\\" in clean or clean.startswith("/") or re.match(r"^[A-Za-z]:", clean)\n            or any(part in ("", ".", "..") for part in clean.split("/"))):\n        raise ValueError("Archive paths must be relative and cannot contain traversal components.")\n    return PurePosixPath(clean).as_posix()\n\n\ndef archive_structures(path: Path, *, max_file_bytes: int, max_total_bytes: int):\n    if path.stat().st_size > MAX_ARCHIVE_BYTES:\n        raise ValueError("Input archives must be at most 200 MB.")\n    seen, count, total = set(), 0, 0\n\n    def admit(name: str, size: int):\n        nonlocal count, total\n        name = safe_archive_name(name)\n        key = name.casefold()\n        if key in seen:\n            raise ValueError("Archive contains duplicate or case-colliding paths.")\n        seen.add(key)\n        count += 1\n        total += size\n        if count > MAX_ARCHIVE_ENTRIES or total > max_total_bytes:\n            raise ValueError("Archive exceeds its entry-count or uncompressed-byte limit.")\n        return name\n\n    if path.suffix.lower() == ".zip":\n        with zipfile.ZipFile(path) as archive:\n            for info in archive.infolist():\n                if info.is_dir() and info.filename in (".", "./"):\n                    continue\n                name = admit(info.filename, info.file_size)\n                kind = stat.S_IFMT(info.external_attr >> 16)\n                if kind not in (0, stat.S_IFREG, stat.S_IFDIR) or info.flag_bits & 1:\n                    raise ValueError("Archives cannot contain links, special files or encrypted entries.")\n                if info.is_dir():\n                    continue\n                if structure_format(name):\n                    if info.file_size > max_file_bytes:\n                        raise ValueError("An archived structure exceeds the allowed byte limit.")\n                    with archive.open(info) as stream:\n                        yield name, read_bounded(stream, max_file_bytes, "Archived structure")\n    else:\n        with tarfile.open(path, "r|*") as archive:\n            for info in archive:\n                if info.isdir() and info.name in (".", "./"):\n                    continue\n                name = admit(info.name, info.size)\n                if not (info.isfile() or info.isdir()):\n                    raise ValueError("Archives cannot contain links or special files.")\n                if info.isfile() and structure_format(name):\n                    if info.size > max_file_bytes:\n                        raise ValueError("An archived structure exceeds the allowed byte limit.")\n                    with archive.extractfile(info) as stream:\n                        yield name, read_bounded(stream, max_file_bytes, "Archived structure")\n\n\ndef is_archive(path: Path) -> bool:\n    return path.name.lower().endswith((".zip", ".tar", ".tar.gz", ".tgz"))\n\n\ndef prepare_inputs(output_zip: str | Path, *, ids: str = "", id_files=(), sources=(),\n                   downloader=download_structure, max_structures: int = MAX_STRUCTURES,\n                   max_file_bytes: int = MAX_FILE_BYTES, max_total_bytes: int = MAX_TOTAL_BYTES,\n                   overwrite: bool = False) -> dict:\n    """Build a complete prepared ZIP atomically; never write extracted members."""\n    if not 1 <= max_structures <= MAX_STRUCTURES or not 1 <= max_file_bytes <= MAX_FILE_BYTES:\n        raise ValueError("Preparation limits cannot exceed 100 structures or 5 MB per structure.")\n    if not 1 <= max_total_bytes <= MAX_TOTAL_BYTES:\n        raise ValueError("Preparation cannot exceed 100 MB of structure data.")\n    output = Path(output_zip).resolve()\n    manifest_path = output.with_suffix(".manifest.json")\n    if output.suffix.lower() != ".zip":\n        raise ValueError("The prepared output must use a .zip extension.")\n    if not overwrite and (output.exists() or manifest_path.exists()):\n        raise FileExistsError("An output already exists; choose another path or use --force.")\n    excluded = {output, manifest_path}\n    if len(ids.encode("utf-8")) > MAX_ID_FILE_BYTES:\n        raise ValueError("Identifier text must be at most 1 MB.")\n    identifiers = parse_ids(ids)\n    local_entries = []\n    collected_bytes = 0\n\n    def collect(name, data, metadata):\n        nonlocal collected_bytes\n        collected_bytes += len(data)\n        if len(local_entries) >= max_structures or collected_bytes > max_total_bytes:\n            raise ValueError("Local inputs exceed the configured structure-count or total-byte limit.")\n        local_entries.append((name, data, metadata))\n\n    for filename in id_files:\n        path = Path(filename)\n        if path.is_symlink() or not path.is_file():\n            raise ValueError("ID-list inputs must be regular files, not links.")\n        with path.open("rb") as stream:\n            identifiers.extend(read_id_list(read_bounded(stream, MAX_ID_FILE_BYTES, "ID list"), path.suffix))\n    files_seen = 0\n    for source in sources:\n        path = Path(source)\n        if path.is_symlink() or not path.exists():\n            raise ValueError("Local inputs must exist and cannot be symbolic links.")\n        if path.is_dir():\n            paths = []\n            for item in path.rglob("*"):\n                files_seen += 1\n                if files_seen > MAX_ARCHIVE_ENTRIES * 10:\n                    raise ValueError("Input folders contain too many entries.")\n                paths.append(item)\n            paths.sort()\n        else:\n            paths = [path]\n        for item in paths:\n            if item.is_symlink():\n                raise ValueError("Input folders cannot contain symbolic links.")\n            if not item.is_file() or item.resolve() in excluded:\n                continue\n            if item.suffix.lower() in (".txt", ".csv", ".tsv") and not path.is_dir():\n                with item.open("rb") as stream:\n                    identifiers.extend(read_id_list(read_bounded(stream, MAX_ID_FILE_BYTES, "ID list"), item.suffix))\n            elif is_archive(item):\n                for name, data in archive_structures(item, max_file_bytes=max_file_bytes, max_total_bytes=max_total_bytes):\n                    collect(name, data, {"source_type": "archive", "archive_name": item.name})\n            elif structure_format(item.name):\n                with item.open("rb") as stream:\n                    collect(item.name, read_bounded(stream, max_file_bytes, "Structure"), {"source_type": "local_file"})\n            elif not path.is_dir():\n                raise ValueError(f"Unsupported input file type: {item.name}")\n    identifiers = list(dict.fromkeys(identifiers))\n    resolved = []\n    seen_urls = set()\n    for identifier in identifiers:\n        metadata = resolve_identifier(identifier)\n        if metadata["source_url"] not in seen_urls:\n            resolved.append((identifier, metadata))\n            seen_urls.add(metadata["source_url"])\n    if len(resolved) + len(local_entries) > max_structures:\n        raise ValueError("Input exceeds the configured structure-count limit.")\n    if not resolved and not local_entries:\n        raise ValueError("Provide at least one supported identifier or structure.")\n    entries, blobs, total_bytes = [], {}, 0\n\n    def add(name: str, data: bytes, metadata: dict):\n        nonlocal total_bytes\n        data = structure_bytes(data, name, max_file_bytes)\n        total_bytes += len(data)\n        if total_bytes > max_total_bytes:\n            raise ValueError("Combined structure data exceeds the configured total-byte limit.")\n        fmt = structure_format(name)\n        original = PurePosixPath(name).name\n        stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", original[:-3] if original.lower().endswith(".gz") else original)\n        stem = Path(stem).stem.strip("._-")[:100] or "structure"\n        archive_path = f"structures/{len(entries) + 1:04d}_{stem}.{fmt}"\n        entry = {"path": archive_path, "original_name": original, "file_format": fmt,\n                 "size_bytes": len(data), "sha256": hashlib.sha256(data).hexdigest(), **metadata}\n        if name != original:\n            entry["archive_member"] = name\n        entries.append(entry)\n        blobs[archive_path] = data\n\n    for identifier, metadata in resolved:\n        add(metadata["filename"], downloader(metadata["source_url"], max_bytes=max_file_bytes),\n            {"source_type": "identifier", "input_id": identifier,\n             "canonical_id": metadata["canonical_id"], "source_url": metadata["source_url"]})\n    for name, data, metadata in local_entries:\n        add(name, data, metadata)\n    manifest = {"schema_version": 1, "prepared_by": "ZincSight Tamarind input preparer",\n                "requires_network_at_runtime": False, "requested_ids": identifiers,\n                "structure_count": len(entries), "total_structure_bytes": total_bytes,\n                "structures": entries}\n    manifest_bytes = (json.dumps(manifest, indent=2, ensure_ascii=True, allow_nan=False) + "\\n").encode()\n    output.parent.mkdir(parents=True, exist_ok=True)\n    temporary_paths = []\n    try:\n        for destination in (output, manifest_path):\n            with tempfile.NamedTemporaryFile(prefix=".zincsight-", dir=output.parent, delete=False) as stream:\n                temporary_paths.append(Path(stream.name))\n        with zipfile.ZipFile(temporary_paths[0], "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:\n            for name, data in [("manifest.json", manifest_bytes), *blobs.items()]:\n                info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))\n                info.compress_type = zipfile.ZIP_DEFLATED\n                info.external_attr = 0o100644 << 16\n                archive.writestr(info, data)\n        if temporary_paths[0].stat().st_size > MAX_ARCHIVE_BYTES:\n            raise ValueError("Prepared ZIP exceeds the 200 MB upload limit.")\n        temporary_paths[1].write_bytes(manifest_bytes)\n        os.replace(temporary_paths[0], output)\n        os.replace(temporary_paths[1], manifest_path)\n    finally:\n        for temporary in temporary_paths:\n            temporary.unlink(missing_ok=True)\n    return manifest\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("sources", nargs="*", help="Structure files, folders, archives, or ID-list files")\n    parser.add_argument("--ids", default="", help="Comma/whitespace-separated public structure identifiers")\n    parser.add_argument("--ids-file", action="append", default=[], help="UTF-8 .txt/.csv/.tsv ID list (repeatable)")\n    parser.add_argument("--output", required=True, type=Path, help="Prepared ZIP to upload as Tamarind\'s batch archive")\n    parser.add_argument("--force", action="store_true", help="Replace an existing prepared output")\n    args = parser.parse_args(argv)\n    try:\n        manifest = prepare_inputs(args.output, ids=args.ids, id_files=args.ids_file,\n                                  sources=args.sources, overwrite=args.force)\n    except (ValueError, OSError, UnicodeError, zipfile.BadZipFile, tarfile.TarError) as exc:\n        parser.exit(2, f"Preparation failed: {exc}\\n")\n    print(f"Prepared {manifest[\'structure_count\']} structures: {args.output}")\n    print(f"Manifest: {args.output.with_suffix(\'.manifest.json\')}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
PREPARER_SHA256 = 'c5ca4a4ef50829106b0660d86b50e9a16b03a58fa768ce5f56cbe10cd8b39795'
assert hashlib.sha256(PREPARER_SOURCE.encode()).hexdigest() == PREPARER_SHA256
preparer = types.ModuleType("zincsight_tamarind_preparer")
preparer.__file__ = "embedded_prepare_inputs.py"
exec(compile(PREPARER_SOURCE, preparer.__file__, "exec"), preparer.__dict__)

ids_form = widgets.Textarea(
    value="", placeholder="1KLS, P12345\n2A0S-assembly1",
    description="IDs:", rows=6, layout=widgets.Layout(width="100%"))
id_files_form = widgets.Checkbox(value=False, description="Upload ID-list files (.txt/.csv/.tsv)", indent=False)
structures_form = widgets.Checkbox(value=False, description="Upload structures or archives", indent=False)
display(widgets.VBox([
    widgets.HTML("<b>Paste IDs separated by commas, spaces, or newlines.</b> Leave blank when uploading files only."),
    ids_form, id_files_form, structures_form,
    widgets.HTML("Edit these inputs, then run <b>2. Prepare ZIP</b> below. No prediction job is submitted.")]))


In [ ]:
#@title 2. Prepare ZIP { display-mode: "form" }
from google.colab import files
import tempfile

work = Path(tempfile.mkdtemp(prefix="zincsight-inputs-", dir="/content"))

def upload_inputs(subfolder, prompt):
    print(prompt)
    uploaded = files.upload(target_dir=str(work / subfolder))
    return [Path(filename) for filename in uploaded]

id_paths = upload_inputs("id_lists", "Choose one or more ID-list files:") if id_files_form.value else []
structure_paths = upload_inputs("structures", "Choose structures and/or archives:") if structures_form.value else []
ids_text = ids_form.value
if not ids_text.strip() and not id_paths and not structure_paths:
    raise ValueError("Enter at least one ID or select an upload option above, then rerun this cell.")
prepared_zip = work / "zincsight_inputs.zip"
manifest = preparer.prepare_inputs(prepared_zip, ids=ids_text, id_files=id_paths, sources=structure_paths)
print(f"Ready: {manifest['structure_count']} structures, {manifest['total_structure_bytes']:,} uncompressed bytes.")
print("Run Download below. Then upload zincsight_inputs.zip to the Tamarind batch-archive input.")


In [ ]:
#@title 3. Download { display-mode: "form" }
from google.colab import files

if "prepared_zip" not in globals() or not prepared_zip.is_file():
    raise ValueError("Run Prepare ZIP successfully before downloading.")
files.download(str(prepared_zip))
print("In Tamarind, open ZincSight Preview, upload this ZIP under Batch archive or prepared ID batch, choose the prediction settings, and run the tool.")
print("The ZIP already contains its provenance manifest; no separate file upload is needed.")
